# Phase 9 — Topic-Aware Classification Colab Runner

This notebook runs Phase 9 on Google Colab L4 using the Google Drive workspace layout:

```text
/content/drive/MyDrive/Authorship-Attribution/
  artifacts/
  backups/
  checkpoints/
  datasets/
  exports/
  repo/
    Authorship-Attribution-in-Victorian-Periodicals/
```

Phase 9 reads:

```text
artifacts/phase8/bertopic/
datasets/processed/periad/train.csv
datasets/processed/periad/test.csv
```

Phase 9 writes:

```text
artifacts/phase9/topic_features/
```

## 0. Select GPU runtime

Before running cells:

`Runtime` → `Change runtime type` → `Hardware accelerator` → `GPU` → preferably **L4**.

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Define workspace paths
from pathlib import Path

WORKSPACE = Path('/content/drive/MyDrive/Authorship-Attribution')
REPO_DIR = WORKSPACE / 'repo' / 'Authorship-Attribution-in-Victorian-Periodicals'
DATASET_TRAIN = WORKSPACE / 'datasets' / 'processed' / 'periad' / 'train.csv'
DATASET_TEST = WORKSPACE / 'datasets' / 'processed' / 'periad' / 'test.csv'
PHASE8_DIR = WORKSPACE / 'artifacts' / 'phase8' / 'bertopic'
PHASE9_DIR = WORKSPACE / 'artifacts' / 'phase9' / 'topic_features'

print('WORKSPACE:', WORKSPACE)
print('REPO_DIR:', REPO_DIR)
print('DATASET_TRAIN:', DATASET_TRAIN)
print('DATASET_TEST:', DATASET_TEST)
print('PHASE8_DIR:', PHASE8_DIR)
print('PHASE9_DIR:', PHASE9_DIR)

In [ ]:
# 4. Clone or update repository
import os, subprocess, textwrap

GITHUB_URL = 'https://github.com/IamTaoHu/Authorship-Attribution-in-Victorian-Periodicals.git'
REPO_PARENT = REPO_DIR.parent
REPO_PARENT.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    print('Repository not found. Cloning...')
    !git clone {GITHUB_URL} {REPO_DIR}
else:
    print('Repository found. Pulling latest main...')
    %cd {REPO_DIR}
    !git status --short
    !git pull

%cd {REPO_DIR}
!git rev-parse --short HEAD

## 5. Install dependencies

This installs the libraries needed for Phase 9. Run this once per Colab session.

In [ ]:
# 5. Install dependencies
!pip install -q -U pip
!pip install -q -r requirements.txt
!pip install -q transformers accelerate evaluate scikit-learn pandas numpy matplotlib seaborn pyyaml joblib

In [ ]:
# 6. Verify required input files and folders
required_paths = [
    DATASET_TRAIN,
    DATASET_TEST,
    PHASE8_DIR,
    PHASE8_DIR / 'tables' / 'document_topic_features.csv',
    REPO_DIR / 'configs' / 'phase9' / 'topic_aware_classification.yaml',
    REPO_DIR / 'scripts' / 'run_phase9_topic_features.py',
    REPO_DIR / 'scripts' / 'check_phase9_outputs.py',
]

missing = [p for p in required_paths if not p.exists()]
if missing:
    print('Missing required paths:')
    for p in missing:
        print(' -', p)
    raise FileNotFoundError('Some required Phase 9 inputs are missing. Check Google Drive paths above.')

print('All required Phase 9 inputs found.')

In [ ]:
# 7. Inspect Phase 8 feature table shape and columns
import pandas as pd

feature_path = PHASE8_DIR / 'tables' / 'document_topic_features.csv'
features_df = pd.read_csv(feature_path)
print('Feature table:', feature_path)
print('Shape:', features_df.shape)
print('Columns:')
print(list(features_df.columns)[:80])
features_df.head()

In [ ]:
# 8. Compile check for Phase 9 code
!python -m py_compile src/classification/topic_features_phase9.py
!python -m py_compile src/visualization/plot_phase9_topic_features.py
!python -m py_compile scripts/run_phase9_topic_features.py
!python -m py_compile scripts/check_phase9_outputs.py
print('Compile checks passed.')

## 9. Optional: topic-only smoke test

This is fast and confirms that Phase 1 + Phase 8 files merge correctly before running DeBERTa.

In [ ]:
# 9. Optional smoke test: topic-only variants
# You can skip this cell if local smoke test already passed.
!python scripts/run_phase9_topic_features.py   --config configs/phase9/topic_aware_classification.yaml   --dataset_train {DATASET_TRAIN}   --dataset_test {DATASET_TEST}   --phase8_dir {PHASE8_DIR}   --output_dir {PHASE9_DIR}   --variants topic_only_logreg,topic_only_linear_svc   --skip_transformers

In [ ]:
# 10. Validate smoke-test outputs
!python scripts/check_phase9_outputs.py   --phase9_dir {PHASE9_DIR}

## 11. Full Phase 9 run on Colab L4

This runs all configured Phase 9 variants, including DeBERTa text-only and topic-aware variants.

Expected output:

```text
/content/drive/MyDrive/Authorship-Attribution/artifacts/phase9/topic_features/
```

In [ ]:
# 11. Full Phase 9 run
!python scripts/run_phase9_topic_features.py   --config configs/phase9/topic_aware_classification.yaml   --dataset_train {DATASET_TRAIN}   --dataset_test {DATASET_TEST}   --phase8_dir {PHASE8_DIR}   --output_dir {PHASE9_DIR}   --variants all

In [ ]:
# 12. Validate full Phase 9 outputs
!python scripts/check_phase9_outputs.py   --phase9_dir {PHASE9_DIR}   --require_models   --require_plots

In [ ]:
# 13. Display result summary tables
import pandas as pd
from IPython.display import display, Markdown

summary_path = PHASE9_DIR / 'tables' / 'phase9_results_summary.csv'
per_author_delta_path = PHASE9_DIR / 'tables' / 'per_author_f1_delta_vs_text_only.csv'
report_path = PHASE9_DIR / 'reports' / 'phase9_topic_aware_classification.md'

summary_df = pd.read_csv(summary_path)
display(Markdown('## Phase 9 Results Summary'))
display(summary_df)

if per_author_delta_path.exists():
    delta_df = pd.read_csv(per_author_delta_path)
    display(Markdown('## Per-author F1 Delta vs Text-only'))
    display(delta_df)

print('Report path:', report_path)
print('Output directory:', PHASE9_DIR)

In [ ]:
# 14. Preview key plots
from IPython.display import Image, display

plot_files = [
    PHASE9_DIR / 'plots' / 'phase9_macro_f1_comparison.png',
    PHASE9_DIR / 'plots' / 'phase9_accuracy_comparison.png',
    PHASE9_DIR / 'plots' / 'per_author_f1_delta_vs_text_only.png',
    PHASE9_DIR / 'plots' / 'confusion_matrix_deberta_topic_concat.png',
]

for plot_path in plot_files:
    if plot_path.exists():
        print(plot_path)
        display(Image(filename=str(plot_path)))
    else:
        print('Missing plot:', plot_path)

In [ ]:
# 15. Print Google Drive output links/paths
print('Phase 9 output folder:')
print(PHASE9_DIR)
print('
Main files:')
for p in [
    PHASE9_DIR / 'tables' / 'phase9_results_summary.csv',
    PHASE9_DIR / 'tables' / 'per_author_f1_comparison.csv',
    PHASE9_DIR / 'tables' / 'per_author_f1_delta_vs_text_only.csv',
    PHASE9_DIR / 'reports' / 'phase9_topic_aware_classification.md',
]:
    print(p)

## Troubleshooting notes

- If merge fails, inspect `sample_id`, `split`, `source_row_id`, and `source_row_index` columns in Phase 1 and Phase 8 files.
- If CUDA memory fails, reduce `per_device_train_batch_size` in `configs/phase9/topic_aware_classification.yaml` and keep `gradient_accumulation_steps` enabled.
- If the full run is interrupted, rerun the full Phase 9 cell. Existing output files may be overwritten by the runner.
- The smoke-test output can be replaced by the full run in the same `artifacts/phase9/topic_features/` directory.